In [ ]:
from qiskit import QuantumRegister, QuantumCircuit, ClassicalRegister
from qiskit.quantum_info import *
from qiskit.visualization import plot_histogram
from IPython.display import display
from qiskit.circuit.library import UnitaryGate
from numpy import pi
from qiskit_aer import AerSimulator
import numpy as np
import math
import random
from fractions import Fraction


# Classical primality test

def is_prime_classical(num):
    if num < 2:
        return False

    if num == 2:
        return True

    if num % 2 == 0:
        return False

    for d in range(3, int(num**0.5) + 1, 2):
        if num % d == 0:
            return False

    return True


# Modular multiplication unitary U_a

def modular_multiplication_unitary(a, N, n):
    """
    Creates the n-qubit unitary U_a:

        U_a |x> = |a*x mod N>,   if 0 <= x < N
        U_a |x> = |x>,           if N <= x < 2^n
    """

    dim = 2**n
    U = np.zeros((dim, dim), dtype=complex)

    for x in range(dim):

        if x < N:
            y = (a * x) % N
        else:
            y = x

        U[y, x] = 1

    return U


# Controlled modular multiplication WITHOUT .control()

def controlled_modular_multiplication_unitary(a_power, N, n):
    """
    Directly creates controlled-U.

    Local qubit order:
        local qubit 0 = control
        local qubits 1...n = target/work register

    Because of Qiskit little-endian convention:

        local basis index = control_bit + 2*x

    If control = 0:
        |0>|x> -> |0>|x>

    If control = 1:
        |1>|x> -> |1>|a_power*x mod N>
    """

    target_dim = 2**n
    total_dim = 2 * target_dim

    CU = np.zeros((total_dim, total_dim), dtype=complex)

    for x in range(target_dim):

        # Control = 0
        
        input_index_0 = 2*x
        output_index_0 = 2*x

        CU[output_index_0, input_index_0] = 1

        # Control = 1
       
        if x < N:
            y = (a_power * x) % N
        else:
            y = x

        input_index_1 = 2*x + 1
        output_index_1 = 2*y + 1

        CU[output_index_1, input_index_1] = 1

    return CU


# Inverse QFT

def apply_inverse_qft(qc, q_reg, m):
    """
    Applies inverse QFT on the first m qubits:
        q_reg[0], q_reg[1], ..., q_reg[m-1]
    """

    # SWAP part
    for i in range(m // 2):
        qc.swap(q_reg[i], q_reg[m - i - 1])

    # Main inverse QFT part
    for i in range(m):
        p = i

        for j in range(p):
            qc.cp(
                -2*pi/(2**(p - j + 1)),
                q_reg[j],
                q_reg[p]
            )

        qc.h(q_reg[p])


# Continued fraction post-processing

def get_order_from_measurement(bitstring, a, N, m):
    """
    From measured bitstring, recover candidate order r.

    Measurement gives integer t such that:

        t / 2^m ≈ j / r

    where r is the order of a modulo N.
    """

    t = int(bitstring, 2)

    print("\n--------------------------------")
    print("Measured bitstring =", bitstring)
    print("Measured integer t =", t)

    if t == 0:
        print("t = 0 gives phase 0, so order cannot be extracted from this result.")
        return None

    phase = t / (2**m)

    print(f"Measured phase ≈ {t}/{2**m} = {phase}")

    frac = Fraction(t, 2**m).limit_denominator(N)

    j_candidate = frac.numerator
    r_candidate = frac.denominator

    print(f"Continued fraction gives approximately: {j_candidate}/{r_candidate}")
    print(f"Candidate order from denominator: r = {r_candidate}")

    if pow(a, r_candidate, N) == 1:
        print(f"Success: {a}^{r_candidate} mod {N} = 1")
        return r_candidate

    print(f"But {a}^{r_candidate} mod {N} = {pow(a, r_candidate, N)}, not 1.")

    print("Checking multiples of candidate denominator...")

    for k in range(2, N + 1):

        possible_r = k * r_candidate

        if possible_r > N:
            break

        if pow(a, possible_r, N) == 1:
            print(f"Success: {a}^{possible_r} mod {N} = 1")
            print(f"Actual order found: r = {possible_r}")
            return possible_r

    print("Failed to extract order from this measurement.")
    return None


# Quantum order-finding circuit

def quantum_order_finding(N, a, shots=2048, show_circuit=False, show_histogram=False):
    """
    Finds the order r of a modulo N using quantum order finding.

    Returns:
        r if found,
        None if failed.
    """

    if math.gcd(a, N) != 1:
        raise ValueError(f"gcd({a},{N}) != 1, so order is not defined.")

    # Choose smallest n such that 2^n >= N
    n = 0
    while 2**n < N:
        n += 1

    # Counting register size
    m = 2*n + 1

    print("\n========== Quantum Order Finding ==========")
    print(f"N = {N}")
    print(f"a = {a}")
    print(f"Smallest n with 2^n >= N: n = {n}")
    print(f"Counting qubits m = 2n + 1 = {m}")

    # Create circuit
    q = QuantumRegister(m + n, "q")
    c = ClassicalRegister(m, "c")

    qc = QuantumCircuit(q, c)

    # Lower register initialized to |1>
    qc.x(q[m])

    # Hadamard on upper/counting register
    for i in range(m):
        qc.h(q[i])

    # Controlled-U operations
    for i in range(m):

        power = 2**i

        # U_a^(2^i) = U_{a^(2^i) mod N}
        a_power = pow(a, power, N)

        CU_power = controlled_modular_multiplication_unitary(a_power, N, n)

        if not np.allclose(CU_power.conj().T @ CU_power, np.eye(2**(n+1))):
            raise ValueError(f"Controlled-U^{power} is not unitary.")

        CU_power_gate = UnitaryGate(CU_power, label=f"c-U^{power}")

        qc.append(
            CU_power_gate,
            [q[i]] + [q[m + j] for j in range(n)]
        )

    # Inverse QFT on upper register
    apply_inverse_qft(qc, q, m)

    # Measure upper register
    for i in range(m):
        qc.measure(q[i], c[i])

    if show_circuit:
        display(qc.draw(output="mpl", style="iqp"))

    # Simulation
    simulator = AerSimulator()

    result = simulator.run(qc, shots=shots).result()
    counts = result.get_counts()

    print("\nMeasurement counts:")
    print(counts)

    if show_histogram:
        display(plot_histogram(counts))

    # Try measurement results from highest count to lowest count
    sorted_counts = sorted(counts.items(), key=lambda item: item[1], reverse=True)

    for bitstring, count in sorted_counts:

        print("\n================================")
        print(f"Trying measured output: {bitstring}")
        print(f"Count: {count}")
        print("================================")

        r_candidate = get_order_from_measurement(bitstring, a, N, m)

        if r_candidate is not None:
            print(f"\nOrder found: r = {r_candidate}")
            return r_candidate

    print("\nOrder finding failed in this run.")
    return None


# One Shor attempt: find one non-trivial factor

def shor_find_one_factor(N, shots=2048, max_attempts=10, show_circuit=False, show_histogram=False):
    """
    Finds one non-trivial factor of N.

    Returns:
        d where 1 < d < N,
        or None if failed.
    """

    if N <= 1:
        return None

    if N % 2 == 0:
        print(f"{N} is even. Factor found: 2")
        return 2

    if is_prime_classical(N):
        print(f"{N} is already prime.")
        return N

    for attempt in range(1, max_attempts + 1):

        print("\n################################")
        print(f"Shor attempt {attempt} for N = {N}")
        print("################################")

        # Choose random x
        x = random.randint(2, N - 2)

        print(f"Randomly chosen x = {x}")

        g = math.gcd(x, N)

        # If gcd is nontrivial, factor found classically
        if g != 1:
            print(f"Classical gcd found factor:")
            print(f"gcd({x},{N}) = {g}")
            return g

        print(f"gcd({x},{N}) = 1, so order finding is needed.")

        # Quantum order finding
        r = quantum_order_finding(
            N,
            x,
            shots=shots,
            show_circuit=show_circuit,
            show_histogram=show_histogram
        )

        if r is None:
            print("Order finding failed. Trying another x.")
            continue

        print(f"\nOrder of {x} modulo {N} is r = {r}")

        # r must be even
        if r % 2 != 0:
            print(f"r = {r} is odd. This x is not useful. Trying another x.")
            continue

        # Compute x^(r/2) mod N
        half_power = pow(x, r // 2, N)

        print(f"x^(r/2) mod N = {x}^{r//2} mod {N} = {half_power}")

        # If x^(r/2) ≡ -1 mod N, fail for this x
        if half_power == N - 1:
            print(f"x^(r/2) ≡ -1 mod N. This x is not useful. Trying another x.")
            continue

        # Try gcds
        factor1 = math.gcd(half_power - 1, N)
        factor2 = math.gcd(half_power + 1, N)

        print(f"gcd(x^(r/2)-1, N) = gcd({half_power}-1, {N}) = {factor1}")
        print(f"gcd(x^(r/2)+1, N) = gcd({half_power}+1, {N}) = {factor2}")

        if factor1 not in [1, N]:
            print(f"Non-trivial factor found: {factor1}")
            return factor1

        if factor2 not in [1, N]:
            print(f"Non-trivial factor found: {factor2}")
            return factor2

        print("Only trivial factors found. Trying another x.")

    print(f"\nFailed to find a non-trivial factor of {N} after {max_attempts} attempts.")
    return None


# Recursive factorization

def shor_recursive_factorization(N, shots=2048, max_attempts=10):
    """
    Recursively factors N into prime factors.

    Returns:
        list of prime factors.
    """

    if N == 1:
        return []

    if is_prime_classical(N):
        return [N]

    if N % 2 == 0:
        return [2] + shor_recursive_factorization(
            N // 2,
            shots=shots,
            max_attempts=max_attempts
        )

    d = shor_find_one_factor(
        N,
        shots=shots,
        max_attempts=max_attempts,
        show_circuit=False,
        show_histogram=False
    )

    if d is None:
        print(f"Could not factor {N}. Returning it as unresolved.")
        return [N]

    if d == N:
        return [N]

    left_factors = shor_recursive_factorization(
        d,
        shots=shots,
        max_attempts=max_attempts
    )

    right_factors = shor_recursive_factorization(
        N // d,
        shots=shots,
        max_attempts=max_attempts
    )

    return left_factors + right_factors


# Main run

N_input = int(input("Enter composite N to factor: "))

print("\n================================")
print(f"Starting Shor factorization for N = {N_input}")
print("================================")

factors = shor_recursive_factorization(
    N_input,
    shots=2048,
    max_attempts=10
)

factors.sort()

print("\n================================")
print(f"Prime factors of {N_input}: {factors}")
print("Check product =", math.prod(factors))
print("================================")


Starting Shor factorization for N = 100

################################
Shor attempt 1 for N = 25
################################
Randomly chosen x = 9
gcd(9,25) = 1, so order finding is needed.

========== Quantum Order Finding ==========
N = 25
a = 9
Smallest n with 2^n >= N: n = 5
Counting qubits m = 2n + 1 = 11

Measurement counts:
{'11001101011': 1, '10110011101': 3, '01001100110': 129, '00000000000': 218, '01001100100': 6, '00110010010': 1, '11001100011': 2, '10110010101': 1, '00110011010': 133, '00110011110': 2, '11001100111': 59, '01001100011': 1, '10110011010': 105, '11101000100': 1, '10110011001': 69, '10011001101': 186, '11001101100': 1, '01001100111': 54, '11001100110': 112, '01100110010': 4, '00011001100': 3, '11010011010': 1, '01100110101': 2, '00011001101': 166, '00110110110': 1, '11100110011': 193, '00110011001': 52, '00110011111': 3, '10000000000': 175, '01100110011': 179, '10011001100': 11, '00011010000': 1, '01001011100': 1, '00011010011': 1, '10110011011': 10, '